In [1]:
import torch
import torch.nn as nn

In [2]:
batch_size = 64
sequence_len = 64
embedding_dim = 32
layer_size = 256
steps = 5000

In [3]:
# input & output initialization
with open("inputs-shakespeare.txt", "r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(chars)}
itoc = dict(enumerate(chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, sequence_len, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(0, sequence_len, 1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, sequence_len, sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(0, sequence_len, sequence_len)

In [4]:
# model declaration
class RecurrentNeuralNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_dim, layer_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.input_linear = nn.Linear(in_features=embedding_dim, out_features=layer_size)
        self.hidden_linear = nn.Linear(in_features=layer_size, out_features=layer_size)
        self.output_linear = nn.Linear(in_features=layer_size, out_features=vocab_size)        

    def forward(self, input, h=None):
        batch_size = input.size(0)
        if h is None:
            h = torch.zeros((batch_size, layer_size))
        logits = []
        for i in range(input.size(1)):
            ch = input[:, i]
            new_h = torch.tanh(self.input_linear(self.embedding(ch)) + self.hidden_linear(h))
            logits.append(self.output_linear(new_h))
            h = new_h
        return torch.stack(logits, dim=1), h

In [5]:
# model initialization
model = RecurrentNeuralNetwork(vocab_size, embedding_dim, layer_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()
    
    ix = torch.randint(0, len(x_train), (batch_size,))
    logits_train, _ = model(x_train[ix])
    loss_train = nn.functional.cross_entropy(logits_train.view(-1, vocab_size), y_train[ix].view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            logits_val, _ = model(x_val)
            loss_val = nn.functional.cross_entropy(logits_val.view(-1, vocab_size), y_val.view(-1))
            print(f"Step {step}: train loss = {loss_train.item()}, val loss = {loss_val.item()}")

    loss_train.backward()
    optimizer.step()

Step 0: train loss = 4.224233627319336, val loss = 4.228605270385742
Step 1000: train loss = 1.5464019775390625, val loss = 1.7151309251785278
Step 2000: train loss = 1.5718382596969604, val loss = 1.6740509271621704
Step 3000: train loss = 1.4033441543579102, val loss = 1.6527029275894165
Step 4000: train loss = 1.4694437980651855, val loss = 1.651986837387085
Step 4999: train loss = 1.4287633895874023, val loss = 1.6460646390914917


In [7]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000
    h = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)            

    while len(output) < max_output_length:
        logits, h = model(i, h)

        P = torch.softmax(logits[0, -1, :], dim=0)
        j = torch.multinomial(P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long)

In [8]:
print(''.join(output))

ROMEO:
Yes, come, Angelo, what's the
sir, I would I side on'; lety
speak.

MARIA:
I have told a great daughter soul.

KING RICHARD III:
So that were tonth,
Palmery be not comes thims, for that I say thou never husband of feeder, awake our would have too much o'rigle her love.

LUCIO:
Sir, though say the enshial bid if this is her too, if I have made a senuted for their son, I'll Montague, thank you.

ISABELLA:
'Tis a right lord, upon us't auffoldsmen.

FRIAR LAURENCE:
'Thie!

LADWICK:
Say the hand with somendry, if applied: whence suit is in honours forsafe you stander, violent four title unachererpire:
A dastiver no more, and Edward death,
I movelve
This, but I'll him should at old than this, my lord.

Provost:
Stand kill.

GREMIO:
What conscience and we will make up them 'tis and ranckess, thou shalt tend you,
God mine;
What first that art revenge
Your noblenedy weep to myself; here the victed, I still:
And make gentlicous will walest leasure my son shall favice shame follow
Doth not